In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f(x, y):
    return -2 * y * np.cos(x)

def analytical_solution(x):
    return np.exp(-2 * np.sin(x))

x_start, x_end = 0, 10 * np.pi

h_values = np.concatenate(([0.01], np.arange(0.05, 1.0, 0.05)))

# store values
valid_h_values = []
avg_errors = []
local_errors_15 = []
avg_y_vals = []
y_at_15_vals = []


plt.figure(figsize=(14, 6))

print(f"{'h':<6} | {'Avg y':<10} | {'y at x≈15':<10} | {'Avg Error':<10} | {'Local Err @ x≈15'}")
print("-" * 65)

for h in h_values:

    num_steps = int(round((x_end - x_start) / h))
    x_values = np.linspace(x_start, x_end, num_steps + 1)
    y_values = np.zeros(num_steps + 1)
    y_values[0] = 1  # Initial condition y(0) = 1

    # Euler's method
    for i in range(num_steps):
        y_values[i + 1] = y_values[i] + h * f(x_values[i], y_values[i])

    # true solution
    y_true = analytical_solution(x_values)
    errors = np.abs(y_values - y_true)

    # 1. Average y value
    avg_y = np.mean(y_values)
    
    # 2. Average Error
    avg_error = np.mean(errors)

    # 3. Values at x ≈ 15
    idx_15 = np.argmin(np.abs(x_values - 15)) # Find index closest to x = 15
    local_error_15 = errors[idx_15]
    y_at_15 = y_values[idx_15]

    # Store values
    valid_h_values.append(h)
    avg_y_vals.append(avg_y)
    avg_errors.append(avg_error)
    local_errors_15.append(local_error_15)
    y_at_15_vals.append(y_at_15)

    # comparison data
    print(f"{h:<6.2f} | {avg_y:<10.4f} | {y_at_15:<10.4f} | {avg_error:<10.4f} | {local_error_15:.4f}")

    if h in [0.01, 0.15, 0.5, 0.95]:
        plt.plot(x_values, y_values, label=f"Euler h={h:.2f}", alpha=0.8)

# Add true analytical solution and plot
x_fine = np.linspace(x_start, x_end, 1000)
plt.plot(x_fine, analytical_solution(x_fine), 'k--', label="True Solution", linewidth=2)
plt.xlabel('x')
plt.ylabel('y')
plt.title("Plot 1: Euler's Method vs True Solution for dy/dx = -2y*cos(x)")
plt.legend()
plt.show()

# Average Error vs h
plt.figure(figsize=(10, 5))
plt.plot(valid_h_values, avg_errors, marker='o', color='b', linestyle='-')
plt.xlabel('Step Size (h)')
plt.ylabel('Average Absolute Error')
plt.title("Plot 2: Average Global Error vs Step Size (h)")
plt.show()

# Local Error at x ≈ 15 vs h 
plt.figure(figsize=(10, 5))
plt.plot(valid_h_values, local_errors_15, marker='o', color='r', linestyle='-')
plt.xlabel('Step Size (h)')
plt.ylabel('Local Absolute Error at x ≈ 15')
plt.title("Plot 3: Local Error at x ≈ 15 vs Step Size (h)")
plt.show()

# -----------Observations-----------

# The true solution to this equation is a repeating, oscillating wave that Euler's method can only 
# track accurately when the step size 'h' is very small. As 'h' increases, Euler's straight-line 
# approximations fail to catch the rapid turns of the wave and severely overshoot the peaks and 
# valleys, which is why the average global error eventually skyrockets out of control. Meanwhile, 
# the local error at a single specific point like x approximately equaling 15 looks much more jagged and random;
# this happens because the oversized steps cause the approximation to drift out of sync with the 
# true wave, meaning that specific point might coincidentally land on a completely wrong part of 
# the cycle regardless of the overall error trend.

# --------addendum------------------


import numpy as np
import matplotlib.pyplot as plt

def f(x, y):
    return -2 * y * np.cos(x)

# Test the boundary (h=1.0) and just past the boundary (h=1.1)
test_hs = [1.0, 1.1]
x_start, x_end = 0, 30

plt.figure(figsize=(10, 5))

for h in test_hs:
    num_steps = int(round((x_end - x_start) / h))
    x_values = np.linspace(x_start, x_end, num_steps + 1)
    y_values = np.zeros(num_steps + 1)
    y_values[0] = 1
    
    for i in range(num_steps):
        y_values[i + 1] = y_values[i] + h * f(x_values[i], y_values[i])
        
    plt.plot(x_values, y_values, marker='o', label=f"h = {h}")
    
    # Print out the maximum value
    print(f"Max y value for h={h}: {np.max(np.abs(y_values)):.2f}")

plt.title("Empirical Stability Test: y' = -2y*cos(x)")
plt.xlabel("x")
plt.ylabel("y")
plt.axhline(0, color='black', linewidth=0.5)
plt.legend()
plt.grid(True)
plt.show()

# ------explanation---------

# To find the stability limit for y' = -2y*cos(x), we need to determine how much the slope depends 
# on y at any given point along the x-axis, which gives us a changing variable lambda(x) = -2cos(x).
# For Euler's method to remain stable and not spiral out to infinity during the phases where the true
# mathematical wave is shrinking, it must satisfy the standard stability condition of |1 + h*lambda| less than or equal to 1.
# When we plug in our specific lambda, the condition becomes |1 - 2h*cos(x)| less than or equal to 1. The ultimate
# "stress test" for the algorithm occurs when the wave is diving downward the fastest, which happens 
# exactly when cos(x) = 1; plugging that worst-case scenario into our inequality gives us |1 - 2h| less than or equalto 1,
# meaning your step size h must strictly be 1 or smaller to keep the approximation from blowing up.